# 17.1 Cost terms

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Piecewise-linearities are often employed to give a more realistic description of costs
than can be achieved by linear terms alone. In this kind of application, piecewise-linear
terms serve much the same purpose as nonlinear ones, but without some of the difficulties
to be described in [Chapter 18](../18/18.md).

To make the comparison explicit, we will use the same transportation example as in
[Chapter 18](../18/18.md). We introduce AMPL's notation for piecewise-linear terms with a simple
example that has a fixed number of cost levels (and linear pieces) for each shipping link.
Then we show how an extension of the notation can use indexing expressions to specify a
varying number of pieces controlled through the data.

**Fixed numbers of pieces**

In a linear transportation model like [Figure 3-1a](../03/3_2_an_AMPL_model_for_the_transportation_problem.ipynb#fig-3-1a), any number of units can be shipped
from a given origin to a given destination at the same cost per unit. More realistically,
however, the most favorable rate may be available for only a limited number of units;
shipments beyond this limit pay higher rates. As an example, imagine that three cost rate
levels are specified for each origin-destination pair. Then the total cost of shipments
along a link increases with the amount shipped in a piecewise-linear fashion, with three
pieces as shown in [Figure 17-1](./17_1_cost_terms.ipynb#fig-17-1).

To model the three-piece costs, we replace the parameter cost of [Figure 3-1a](../03/3_2_an_AMPL_model_for_the_transportation_problem.ipynb#fig-3-1a) by
three rates and two limits:

```ampl
param rate1 {i in ORIG, j in DEST} >= 0;
param rate2 {i in ORIG, j in DEST} >= rate1[i,j];
param rate3 {i in ORIG, j in DEST} >= rate2[i,j];
param limit1 {i in ORIG, j in DEST} > 0;
param limit2 {i in ORIG, j in DEST} > limit1[i,j];
```

Shipments from `i` to `j` are charged at rate1[i,j] per unit up to limit1[i,j]
units, then at rate2[i,j] per unit up to limit2[i,j], and then at rate3[i,j].
Normally rate2[i,j] would be greater than rate1[i,j] and rate3[i,j] would
be greater than rate2[i,j], but they may be equal if the link from `i` to `j` does not
have three distinct rates.

In the linear transportation model, the objective is expressed in terms of the variables
and the parameter cost as follows:

```ampl
var Trans {ORIG,DEST} >= 0;
minimize Total_Cost:
       sum {i in ORIG, j in DEST} cost[i,j] * Trans[i,j];
```

We could express a piecewise-linear objective analogously, by introducing three collections
of variables, one to represent the amount shipped at each rate:

```ampl
var Trans1 {i in ORIG, j in DEST} >= 0, <= limit1[i,j];
var Trans2 {i in ORIG, j in DEST} >= 0, <= limit2[i,j]
                                                 - limit1[i,j];
var Trans3 {i in ORIG, j in DEST} >= 0;
minimize Total_Cost:
       sum {i in ORIG, j in DEST} (rate1[i,j] * Trans1[i,j]
       + rate2[i,j] * Trans2[i,j] + rate3[i,j] * Trans3[i,j]);
```

But then the new variables would have to be introduced into all the constraints, and we
would also have to deal with these variables whenever we wanted to display the optimal
results. Rather than go to all this trouble, we would much prefer to describe the
piecewise-linear cost function explicitly in terms of the original variables. Since there is
no standard way to describe piecewise-linear functions in algebraic notation, AMPL supplies
its own syntax for this purpose.

The piecewise-linear function depicted in [Figure 17-1](./17_1_cost_terms.ipynb#fig-17-1) is written in AMPL as follows:

```ampl
<<limit1[i,j], limit2[i,j];
       rate1[i,j], rate2[i,j], rate3[i,j]>> Trans[i,j]
```

The expression between << and >> describes the piecewise-linear function, and is followed
by the name of the variable to which it applies. (You can think of it as "multiplying
" Trans[i,j], but by a series of coefficients rather than just one.) There are two
parts to the expression, a list of breakpoints where the slope of the function changes, and
a list of the slopes — which in this case are the cost rates. The lists are separated by a
semicolon, and members of each list are separated by commas. Since the first slope
applies to values before the first breakpoint, and the last slope to values after the last
breakpoint, the number of slopes must be one more than the number of breakpoints.

Although the lists of breakpoints and slopes are sufficient to describe the piecewiselinear
cost function for optimization, they do not quite specify the function uniquely. If
we added, say, 10 to the cost at every point, we would have a different cost function even
though all the breakpoints and slopes would be the same. To resolve this ambiguity,

```ampl
set ORIG;                       # origins
set DEST;                       # destinations
param supply {ORIG} >= 0;       # amounts available at origins
param demand {DEST} >= 0;       # amounts required at destinations
              check: sum {i in ORIG} supply[i] = sum {j in DEST} demand[j];
param rate1 {i in ORIG, j in DEST} >= 0;
param rate2 {i in ORIG, j in DEST} >= rate1[i,j];
param rate3 {i in ORIG, j in DEST} >= rate2[i,j];
param limit1 {i in ORIG, j in DEST} > 0;
param limit2 {i in ORIG, j in DEST} > limit1[i,j];
var Trans {ORIG,DEST} >= 0;     # units to be shipped
minimize Total_Cost:
       sum {i in ORIG, j in DEST}
       <<limit1[i,j], limit2[i,j];
              rate1[i,j], rate2[i,j], rate3[i,j]>> Trans[i,j];
subject to Supply {i in ORIG}:
       sum {j in DEST} Trans[i,j] = supply[i];
subject to Demand {j in DEST}:
       sum {i in ORIG} Trans[i,j] = demand[j];
```

**[Figure 17-2](./17_1_cost_terms.ipynb#fig-17-2):** Piecewise-linear model with three slopes (transpl1.mod).

AMPL assumes that a piecewise-linear function evaluates to zero at zero, as in Figure
**17-1.** Options for other possibilities are discussed later in this chapter.

Summing the cost over all links, the piecewise-linear objective function is now written

```ampl
minimize Total_Cost:
       sum {i in ORIG, j in DEST}
       <<limit1[i,j], limit2[i,j];
              rate1[i,j], rate2[i,j], rate3[i,j]>> Trans[i,j];
```

The declarations of the variables and constraints stay the same as before; the complete
model is shown in [Figure 17-2](./17_1_cost_terms.ipynb#fig-17-2).

**Varying numbers of pieces**

The approach taken in the preceding example is most useful when there are only a few
linear pieces for each term. If there were, for example, 12 pieces instead of three, a
model defining rate1[i,j] through rate12[i,j] and limit1[i,j] through
limit11[i,j] would be unwieldy. Realistically, moreover, there would more likely
be up to 12 pieces, rather than exactly 12, for each term; a term with fewer than 12 pieces
could be handled by making some rates equal, but for large numbers of pieces this would
be a cumbersome device that would require many unnecessary data values and would
obscure the actual number of pieces in each term.
A much better approach is to let the number of pieces (that is, the number of shipping
rates) itself be a parameter of the model, indexed over the links:

```ampl
param npiece {ORIG,DEST} integer >= 1;
```

We can then index the rates and limits over all combinations of links and pieces:

```ampl
param rate {i in ORIG, j in DEST, p in 1..npiece[i,j]}
       >= if p = 1 then 0 else rate[i,j,p-1];
param limit {i in ORIG, j in DEST, p in 1..npiece[i,j]-1}
       > if p = 1 then 0 else limit[i,j,p-1];
```

For any particular origin `i` and destination j, the number of linear pieces in the cost term
is given by npiece[i,j]. The slopes are rate[i,j,p] for `p` ranging from 1 to
npiece[i,j], and the intervening breakpoints are limit[i,j,p] for `p` from 1 to
npiece[i,j]-1. As before, there is one more slope than there are breakpoints.

To use AMPL's piecewise-linear function notation with these data values, we have to
give indexed lists of breakpoints and slopes, rather than the explicit lists of the previous
example. This is done by placing indexing expressions in front of the slope and breakpoint
values:

```ampl
minimize Total_Cost:
       sum {i in ORIG, j in DEST}
       <<{p in 1..npiece[i,j]-1} limit[i,j,p];
              {p in 1..npiece[i,j]} rate[i,j,p]>> Trans[i,j];
```

Once again, the rest of the model is the same. [Figure 17-3a](./17_1_cost_terms.ipynb#fig-17-3a) shows the whole model and
[Figure 17-3b](./17_1_cost_terms.ipynb#fig-17-3b) illustrates how the data would be specified. Notice that since
npiece["PITT","STL"] is 1, Trans["PITT","STL"] has only one slope and
no breakpoints; this implies a one-piece linear term for Trans["PITT","STL"] in
the objective function.